In [ ]:
pip install yake

In [8]:
import os
import pandas as pd
import yake

base_folder = 'IMS2013-2024'

# Создаём два экстрактора для n=1 и n=2
kw_extractor_1 = yake.KeywordExtractor(lan="ru", n=1, top=20)
kw_extractor_2 = yake.KeywordExtractor(lan="ru", n=2, top=20)

results = []
index_counter = 1
top_n = 10

for year_folder in os.listdir(base_folder):
    year_path = os.path.join(base_folder, year_folder)

    if os.path.isdir(year_path):
        for filename in os.listdir(year_path):
            file_path = os.path.join(year_path, filename)

            if (
                os.path.isfile(file_path) and
                filename.endswith(".txt") and
                "Abstract" not in filename and
                "KW" not in filename
            ):
                parts = filename.split("_")

                try:
                    ims_index = parts.index("IMS")
                    author = "_".join(parts[:ims_index])
                    year_part = parts[ims_index + 1]
                    year = ''.join(filter(str.isdigit, year_part))
                except (ValueError, IndexError):
                    continue

                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        text = f.read()
                except Exception:
                    text = ""

                # Извлекаем ключевые выражения с n=1 и n=2
                try:
                    kw_1 = kw_extractor_1.extract_keywords(text)
                    kw_2 = kw_extractor_2.extract_keywords(text)

                    # Объединяем, убираем дубликаты, сортируем по score
                    combined = kw_1 + kw_2
                    unique_kw = {}
                    for kw, score in combined:
                        if kw not in unique_kw or score < unique_kw[kw]:
                            unique_kw[kw] = score

                    sorted_kw = sorted(unique_kw.items(), key=lambda x: x[1])
                    top_keywords = [kw for kw, _ in sorted_kw[:top_n]]
                    kw_string = ", ".join(top_keywords)
                except Exception:
                    kw_string = ""

                results.append({
                    "Index": index_counter,
                    "Year": year,
                    "Name": author,
                    "KW": kw_string
                })

                index_counter += 1

if results:
    df = pd.DataFrame(results)
    df.to_csv("YAKE_Corp.csv", index=False)

In [10]:
import os
import pandas as pd
import yake

base_folder = 'IMS2013-2024'

# Создаём два экстрактора для n=1 и n=2
kw_extractor_1 = yake.KeywordExtractor(lan="ru", n=1, top=20)
kw_extractor_2 = yake.KeywordExtractor(lan="ru", n=2, top=20)

results = []
index_counter = 1
top_n = 10

for year_folder in os.listdir(base_folder):
    year_path = os.path.join(base_folder, year_folder)

    if os.path.isdir(year_path):
        for filename in os.listdir(year_path):
            file_path = os.path.join(year_path, filename)

            # Ищем только файлы с "KW" в названии, исключая "Abstract" и .txt файлы
            if (
                os.path.isfile(file_path) and
                "KW" in filename and
                "Abstract" not in filename and
                not filename.endswith(".txt")
            ):
                parts = filename.split("_")

                try:
                    ims_index = parts.index("IMS")
                    author = "_".join(parts[:ims_index])
                    year_part = parts[ims_index + 1]
                    year = ''.join(filter(str.isdigit, year_part))
                except (ValueError, IndexError):
                    continue

                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        manual_kw = f.read().strip()  # Ключевые слова из KW файла
                except Exception:
                    manual_kw = ""

                # Находим соответствующий текстовый файл для извлечения ключевых слов через YAKE
                txt_filename = filename.replace("KW", "").strip("_") + ".txt"
                txt_file_path = os.path.join(year_path, txt_filename)
                
                yake_kw = ""
                if os.path.exists(txt_file_path):
                    try:
                        with open(txt_file_path, 'r', encoding='utf-8') as f:
                            text = f.read()
                        
                        # Извлекаем ключевые выражения с n=1 и n=2
                        kw_1 = kw_extractor_1.extract_keywords(text)
                        kw_2 = kw_extractor_2.extract_keywords(text)

                        # Объединяем, убираем дубликаты, сортируем по score
                        combined = kw_1 + kw_2
                        unique_kw = {}
                        for kw, score in combined:
                            if kw not in unique_kw or score < unique_kw[kw]:
                                unique_kw[kw] = score

                        sorted_kw = sorted(unique_kw.items(), key=lambda x: x[1])
                        top_keywords = [kw for kw, _ in sorted_kw[:top_n]]
                        yake_kw = ", ".join(top_keywords)
                    except Exception:
                        yake_kw = ""

                # Добавляем сравнение в результаты
                results.append({
                    "Index": index_counter,
                    "Year": year,
                    "Name": author,
                    "Manual_KW": manual_kw,
                    "YAKE_KW": yake_kw,
                    "Match": manual_kw.lower() == yake_kw.lower()  # Простое сравнение строк
                })

                index_counter += 1

if results:
    df = pd.DataFrame(results)
    # Добавляем более сложное сравнение (сколько ключевых слов совпадают)
    df['Manual_KW_List'] = df['Manual_KW'].str.split(', ')
    df['YAKE_KW_List'] = df['YAKE_KW'].str.split(', ')
    df['Intersection'] = df.apply(
        lambda x: list(set(x['Manual_KW_List']) & set(x['YAKE_KW_List'])), 
        axis=1
    )
    df['Intersection_Count'] = df['Intersection'].apply(len)
    df['Intersection_Ratio'] = df['Intersection_Count'] / df['Manual_KW_List'].apply(len)
    
    df.to_csv("YAKE_vs_Manual_KW_Comparison.csv", index=False)

In [11]:
import os
import pandas as pd
import yake
from collections import defaultdict

base_folder = 'IMS2013-2024'

# Инициализация YAKE
kw_extractor_1 = yake.KeywordExtractor(lan="ru", n=1, top=20)
kw_extractor_2 = yake.KeywordExtractor(lan="ru", n=2, top=20)

results = []
index_counter = 1
top_n = 10

# Сначала соберем все файлы и их типы
file_registry = defaultdict(dict)

for year_folder in os.listdir(base_folder):
    year_path = os.path.join(base_folder, year_folder)
    
    if os.path.isdir(year_path):
        for filename in os.listdir(year_path):
            file_path = os.path.join(year_path, filename)
            
            if not os.path.isfile(file_path):
                continue
                
            # Парсим базовое имя файла (без KW/Abstract)
            base_name = None
            if "KW" in filename:
                base_name = filename.replace("KW", "").strip("_")
                file_registry[base_name]['kw_file'] = file_path
            elif "Abstract" in filename:
                continue  # пропускаем абстракты
            elif filename.endswith(".txt"):
                base_name = filename.replace(".txt", "")
                file_registry[base_name]['txt_file'] = file_path

# Обрабатываем найденные файлы
for base_name, files in file_registry.items():
    parts = base_name.split("_")
    
    try:
        ims_index = parts.index("IMS")
        author = "_".join(parts[:ims_index])
        year_part = parts[ims_index + 1]
        year = ''.join(filter(str.isdigit, year_part))
    except (ValueError, IndexError):
        continue
    
    manual_kw = ""
    yake_kw = ""
    
    # Чтение ручных ключевых слов (если есть)
    if 'kw_file' in files:
        try:
            with open(files['kw_file'], 'r', encoding='utf-8') as f:
                manual_kw = f.read().strip()
        except Exception:
            pass
    
    # Извлечение ключевых слов YAKE (если есть txt файл)
    if 'txt_file' in files:
        try:
            with open(files['txt_file'], 'r', encoding='utf-8') as f:
                text = f.read()
            
            kw_1 = kw_extractor_1.extract_keywords(text)
            kw_2 = kw_extractor_2.extract_keywords(text)
            
            combined = kw_1 + kw_2
            unique_kw = {}
            for kw, score in combined:
                if kw not in unique_kw or score < unique_kw[kw]:
                    unique_kw[kw] = score
            
            sorted_kw = sorted(unique_kw.items(), key=lambda x: x[1])
            top_keywords = [kw for kw, _ in sorted_kw[:top_n]]
            yake_kw = ", ".join(top_keywords)
        except Exception:
            pass
    
    # Добавляем запись только если есть хотя бы один набор ключевых слов
    if manual_kw or yake_kw:
        # Преобразуем в списки для сравнения
        manual_list = [kw.strip() for kw in manual_kw.split(",")] if manual_kw else []
        yake_list = [kw.strip() for kw in yake_kw.split(",")] if yake_kw else []
        
        # Вычисляем метрики сравнения
        intersection = list(set(manual_list) & set(yake_list))
        intersection_count = len(intersection)
        manual_count = len(manual_list)
        yake_count = len(yake_list)
        
        # Избегаем деления на ноль
        ratio_manual = intersection_count / manual_count if manual_count > 0 else 0
        ratio_yake = intersection_count / yake_count if yake_count > 0 else 0
        
        results.append({
            "Index": index_counter,
            "Year": year,
            "Name": author,
            "Manual_KW": manual_kw,
            "Manual_Count": manual_count,
            "YAKE_KW": yake_kw,
            "YAKE_Count": yake_count,
            "Intersection": ", ".join(intersection),
            "Intersection_Count": intersection_count,
            "Ratio_Manual": ratio_manual,
            "Ratio_YAKE": ratio_yake,
            "Status": "Both" if manual_kw and yake_kw else 
                     "Manual Only" if manual_kw else 
                     "YAKE Only"
        })
        
        index_counter += 1

# Сохраняем результаты
if results:
    df = pd.DataFrame(results)
    # Сортируем по году и имени
    df = df.sort_values(by=['Year', 'Name'])
    
    # Сохраняем в CSV с дополнительной информацией
    output_file = "KW_Comparison_Results.csv"
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"Результаты сохранены в {output_file}")
    
    # Генерируем сводную статистику
    stats = {
        "Total_Records": len(df),
        "Both_KW_Present": len(df[df['Status'] == "Both"]),
        "Manual_Only": len(df[df['Status'] == "Manual Only"]),
        "YAKE_Only": len(df[df['Status'] == "YAKE Only"]),
        "Average_Intersection_Ratio": df[df['Status'] == "Both"]['Ratio_Manual'].mean(),
        "Average_Manual_Count": df['Manual_Count'].mean(),
        "Average_YAKE_Count": df['YAKE_Count'].mean()
    }
    
    stats_df = pd.DataFrame([stats])
    stats_df.to_csv("KW_Comparison_Stats.csv", index=False)
    print("Сводная статистика сохранена в KW_Comparison_Stats.csv")

Результаты сохранены в KW_Comparison_Results.csv
Сводная статистика сохранена в KW_Comparison_Stats.csv
